# 解答例：乗降CAの課題

元Notebook：[31_train_boarding_ca.ipynb](../31_train_boarding_ca.ipynb)．

課題，解答，実行に必要な定義だけをまとめている．上から単独で実行できる．具体的な代替行動の実装と優劣の結論は，自分の研究の問いに応じて検討する．

## 課題：実装と検証

1. `door_width_sweep(door_halves, seeds)` を作り，降車8人・乗車0人を基準に幅ごとの全試行を保存する．終了状態，完了時刻，上限制限時間 `capped_time`，残人数を記録し，完了率と時間の要約を表にする．
2. `passenger_count_sweep(counts, seeds)` を作り，降車8人を固定して乗車人数だけを変える．未完了の時刻を完了時間として平均せず，共通上限 `C` での `mean(min(T,C))` と完了率を示す．人数差が不均等でも比較できるように，平均の差を人数差で割った列を加える．
3. 初期配置の保存・複製を実装し，固定障害物の有無で同じ配置を使う．障害物を置く候補セルは両条件の初期配置から除外し，人数を変えない．
4. 1人の最短経路，2人の競合，向かい合う2人の停止を小配置で検証する．上限と同じ時刻の完了を誤って打切りとしないことも `assert` で確認する．
5. 1回の結果や少数条件から単調性を断定せず，完了率，未完了の状態，時間の変化を分けて説明する．

## 実行に必要な定義

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
from copy import deepcopy

def make_grid(H=11, W=21, door_col=10, door_half=2, obstacles=()):
    values = [H, W, door_col, door_half]
    if any(not isinstance(v, (int, np.integer)) for v in values):
        raise ValueError('grid dimensions and door indices must be integers')
    if H < 3 or W < 3 or not 0 < door_col < W-1 or not 0 <= door_half <= min(H//2, H-1-H//2):
        raise ValueError('invalid grid or door width')
    passable = np.ones((H, W), dtype=bool)
    passable[:, door_col] = False
    passable[H//2-door_half:H//2+door_half+1, door_col] = True
    for pos in obstacles:
        if len(pos) != 2 or any(not isinstance(v, (int, np.integer)) for v in pos):
            raise ValueError('obstacle positions must be integer (row, column) pairs')
        r, c = pos
        if not (0 <= r < H and 0 <= c < W) or not passable[r, c]:
            raise ValueError('obstacle must occupy a valid floor cell')
        passable[r, c] = False
    return passable


def neighbors(pos, shape):
    r, c = pos
    return [(nr, nc) for nr, nc in [(r-1, c), (r+1, c), (r, c-1), (r, c+1)]
            if 0 <= nr < shape[0] and 0 <= nc < shape[1]]


def distance_field(passable, goal_col):
    # 全ゴールから幅優先探索する．人を除き，壁・固定障害物は考慮する．
    distance = np.full(passable.shape, np.inf)
    queue = deque()
    for row in range(passable.shape[0]):
        if passable[row, goal_col]:
            distance[row, goal_col] = 0
            queue.append((row, goal_col))
    while queue:
        pos = queue.popleft()
        for target in neighbors(pos, passable.shape):
            if passable[target] and np.isinf(distance[target]):
                distance[target] = distance[pos] + 1
                queue.append(target)
    return distance


def place_agents(passable, door_col, n_alight, n_board, seed=0, excluded=()):
    if any(not isinstance(n, (int, np.integer)) or n < 0 for n in [n_alight, n_board]):
        raise ValueError('passenger counts must be nonnegative integers')
    rng = np.random.default_rng(seed)
    excluded = set(map(tuple, excluded))
    agents = []
    for kind, count, columns in [('alight', n_alight, range(door_col)),
                                  ('board', n_board, range(door_col+1, passable.shape[1]))]:
        cells = [(r, c) for r in range(passable.shape[0]) for c in columns
                 if passable[r, c] and (r, c) not in excluded]
        if count > len(cells):
            raise ValueError('requested passenger count exceeds available cells')
        order = rng.permutation(len(cells))
        for index in order[:count]:
            agents.append(dict(id=len(agents), type=kind, pos=cells[index]))
    return agents


def occupied_cells(agents):
    return {a['pos'] for a in agents if not a['done']}


def check_state(agents, passable):
    active = [a for a in agents if not a['done']]
    assert len({a['id'] for a in agents}) == len(agents)
    assert len(occupied_cells(agents)) == len(active), 'overlapping passengers'
    assert all(passable[a['pos']] for a in active), 'passenger in wall/obstacle'


def propose_moves(agents, passable, fields, rng):
    # 全員が同じ更新前の占有状態を見る．占有セルへの追従・交換はしない．
    occupied = occupied_cells(agents)
    proposals = {}
    for a in agents:
        if a['done']:
            continue
        distance = fields[a['type']]
        options = [pos for pos in neighbors(a['pos'], passable.shape)
                   if passable[pos] and pos not in occupied and distance[pos] < distance[a['pos']]]
        if options:
            best = min(distance[pos] for pos in options)
            options = [pos for pos in options if distance[pos] == best]
            proposals[a['id']] = options[int(rng.integers(len(options)))]
    return proposals


def resolve_proposals(proposals, rng):
    by_target = {}
    for agent_id, target in proposals.items():
        by_target.setdefault(target, []).append(agent_id)
    winners = {}
    conflict_cells = 0
    for target, ids in sorted(by_target.items()):
        conflict_cells += int(len(ids) > 1)
        winner = ids[int(rng.integers(len(ids)))]
        winners[winner] = target
    return winners, conflict_cells


def simulate(H=11, W=21, door_col=10, door_half=2, n_alight=8, n_board=0,
             obstacles=(), max_steps=300, seed=0, initial_agents=None):
    if not isinstance(max_steps, (int, np.integer)) or max_steps < 1:
        raise ValueError('max_steps must be a positive integer')
    if any(not isinstance(n, (int, np.integer)) or n < 0 for n in [n_alight, n_board]):
        raise ValueError('passenger counts must be nonnegative integers')
    passable = make_grid(H, W, door_col, door_half, obstacles)
    # 32章の子SeedSequenceも受け取り，入力のspawn状態は変更しない．
    sequence = (np.random.SeedSequence(seed.entropy, spawn_key=seed.spawn_key, pool_size=seed.pool_size)
                if isinstance(seed, np.random.SeedSequence) else np.random.SeedSequence(seed))
    init_seed, update_seed = sequence.spawn(2)
    rng = np.random.default_rng(update_seed)
    initial = (place_agents(passable, door_col, n_alight, n_board, seed=init_seed)
               if initial_agents is None else deepcopy(initial_agents))
    if len(initial) != n_alight+n_board or len({a['id'] for a in initial}) != len(initial):
        raise ValueError('initial counts or IDs are inconsistent')
    if any(a['type'] not in ('alight', 'board') for a in initial):
        raise ValueError('unknown passenger type')
    if sum(a['type'] == 'alight' for a in initial) != n_alight:
        raise ValueError('initial passenger types do not match counts')
    fields = {'alight': distance_field(passable, W-1), 'board': distance_field(passable, 0)}
    agents = []
    for source in initial:
        pos = tuple(source['pos'])
        if len(pos) != 2 or any(not isinstance(v, (int, np.integer)) for v in pos):
            raise ValueError('passenger positions must be integer pairs')
        if not (0 <= pos[0] < H and 0 <= pos[1] < W) or not passable[pos]:
            raise ValueError('invalid initial position')
        d = fields[source['type']][pos]
        if not np.isfinite(d):
            raise ValueError('initial position cannot reach its goal through the static geometry')
        agents.append(dict(id=source['id'], type=source['type'], pos=pos, done=(d == 0),
                           finished_at=0 if d == 0 else None, moves=0, wait_steps=0))
    if len(occupied_cells(agents)) != sum(not a['done'] for a in agents):
        raise ValueError('overlapping initial passengers')
    # 完了済みも含め，初期配置の重複は許さない．
    if len({tuple(a['pos']) for a in initial}) != len(initial):
        raise ValueError('overlapping initial passengers')
    counts = np.zeros((H, W), dtype=int)
    history = []
    t = 0
    total_conflicts = 0
    status = 'cutoff'

    def record(moved=0, conflicts=0):
        history.append(dict(t=t, remaining_alight=sum(not a['done'] and a['type'] == 'alight' for a in agents),
                            remaining_board=sum(not a['done'] and a['type'] == 'board' for a in agents),
                            moved=moved, conflict_cells=conflicts))

    record()
    while t < max_steps:
        check_state(agents, passable)
        if all(a['done'] for a in agents):
            status = 'completed'
            break
        proposals = propose_moves(agents, passable, fields, rng)
        if not proposals:
            # この基準規則には自発移動・確率的休止がないため吸収状態と証明できる．
            status = 'deadlock'
            for a in agents:
                if not a['done']:
                    counts[a['pos']] += max_steps-t
                    a['wait_steps'] += max_steps-t
            break
        winners, conflicts = resolve_proposals(proposals, rng)
        for a in agents:
            if not a['done']:
                counts[a['pos']] += 1  # 時刻 t の占有を記録してから更新する．
                if a['id'] in winners:
                    a['pos'] = winners[a['id']]
                    a['moves'] += 1
                    if fields[a['type']][a['pos']] == 0:
                        a['done'], a['finished_at'] = True, t+1
                else:
                    a['wait_steps'] += 1
        t += 1
        total_conflicts += conflicts
        check_state(agents, passable)
        record(len(winners), conflicts)
    # 上限と同じ時刻に完了した試行も正常完了である．
    if all(a['done'] for a in agents):
        status = 'completed'
    completion = float(t) if status == 'completed' else np.nan

    def group_finish(kind):
        group = [a for a in agents if a['type'] == kind]
        return (float(max((a['finished_at'] for a in group), default=0))
                if all(a['done'] for a in group) else np.nan)

    return dict(status=status, completed=status == 'completed', cutoff=status == 'cutoff',
                completion_time=completion, capped_time=completion if status == 'completed' else float(max_steps),
                steps_evaluated=t, horizon=max_steps, remaining=sum(not a['done'] for a in agents),
                alight_finished=group_finish('alight'), board_finished=group_finish('board'),
                agents=agents, initial_agents=deepcopy(initial), history=pd.DataFrame(history),
                occupancy_counts=counts, occupancy_fraction=counts/max_steps, passable=passable,
                fields=fields, conflict_cells=total_conflicts)

## 解答例：幅と人数の反復比較

完了者だけの統計には `completed_only` を付ける．未完了を含む条件の全体平均と解釈しない．全試行から得られる完了率と上限制限平均も同時に残す．同じseedでも人数が異なれば初期状態は同一ではない．

In [2]:
def summarize_trials(raw, variable):
    rows = []
    for value, group in raw.groupby(variable, sort=True):
        completed = group.loc[group['completed'], 'completion_time']
        rows.append({variable: value, 'trials': len(group), 'completed': len(completed),
                     'completion_rate': group['completed'].mean(),
                     'completed_only_mean': completed.mean(),
                     'completed_only_std': completed.std(ddof=1),
                     'completed_only_max': completed.max(),
                     'mean_capped': group['capped_time'].mean()})
    return pd.DataFrame(rows)


def door_width_sweep(door_halves, seeds):
    rows = []
    seeds = list(seeds)
    for half in door_halves:
        for seed in seeds:
            result = simulate(door_half=half, n_alight=8, n_board=0, seed=seed)
            rows.append(dict(door_width=2*half+1, seed=seed, **{k: result[k] for k in
                ['completion_time', 'completed', 'status', 'capped_time', 'horizon', 'remaining']}))
    raw = pd.DataFrame(rows)
    return raw, summarize_trials(raw, 'door_width')


def passenger_count_sweep(counts, seeds):
    rows = []
    seeds = list(seeds)
    for count in counts:
        for seed in seeds:
            result = simulate(n_alight=8, n_board=count, seed=seed)
            rows.append(dict(n_board=count, seed=seed, **{k: result[k] for k in
                ['completion_time', 'completed', 'status', 'capped_time', 'horizon', 'remaining']}))
    raw = pd.DataFrame(rows)
    summary = summarize_trials(raw, 'n_board')
    summary['slope_capped'] = summary['mean_capped'].diff()/summary['n_board'].diff()
    return raw, summary


width_raw, width_summary = door_width_sweep([0, 1, 2], range(20))
count_raw, count_summary = passenger_count_sweep([0, 1, 2, 4, 8], range(20))
display(width_summary)
display(count_summary)

,door_width,trials,completed,completion_rate,completed_only_mean,completed_only_std,completed_only_max,mean_capped
0,1,20,20,1.0,27.70,1.490320,31.0,27.70
1,3,20,20,1.0,22.10,1.483240,24.0,22.10
2,5,20,20,1.0,21.05,1.503505,23.0,21.05


,n_board,trials,completed,completion_rate,completed_only_mean,completed_only_std,completed_only_max,mean_capped,slope_capped
0,0,20,20,1.00,21.05,1.503505,23.0,21.05,NaN
1,1,20,5,0.25,21.60,0.894427,23.0,230.40,209.35
2,2,20,3,0.15,22.00,1.000000,23.0,258.30,27.90
3,4,20,0,0.00,NaN,NaN,NaN,300.00,20.85
4,8,20,0,0.00,NaN,NaN,NaN,300.00,0.00


## 解答例：同じ初期配置と小配置検証

In [3]:
obstacles = [(4, 11), (6, 11)]
initial = place_agents(make_grid(), 10, 8, 0, seed=15, excluded=obstacles)
saved = deepcopy(initial)
base = simulate(initial_agents=initial)
alt = simulate(initial_agents=initial, obstacles=obstacles)
assert initial == saved  # simulate は入力を変更しない．
assert base['initial_agents'] == alt['initial_agents']
assert all(a['pos'] not in obstacles for a in initial)

one = [dict(id=0, type='alight', pos=(2, 1))]
settings = dict(H=5, W=5, door_col=2, door_half=0, n_alight=1, n_board=0, initial_agents=one)
assert simulate(**settings, max_steps=3)['status'] == 'completed'
assert simulate(**settings, max_steps=2)['status'] == 'cutoff'
winners, conflicts = resolve_proposals({0: (2, 2), 1: (2, 2)}, np.random.default_rng(0))
assert len(winners) == 1 and conflicts == 1
opposing = [dict(id=0, type='alight', pos=(2, 1)), dict(id=1, type='board', pos=(2, 3))]
jam = simulate(H=5, W=5, door_col=2, door_half=0, n_alight=1, n_board=1, initial_agents=opposing)
assert jam['status'] == 'deadlock' and np.isnan(jam['completion_time'])
assert jam['capped_time'] == jam['horizon']
print('initial-state pairing, completion boundary, conflict, and deadlock checks passed')

initial-state pairing, completion boundary, conflict, and deadlock checks passed


## 解答例：結果の解釈

このモデルの一方向流では，誰かの残り距離が必ず減り，十分長い上限なら全員が完了する．対向流では，距離を減らす移動だけでは避けられない状態が生じる．人数増加による上限制限平均の増加は，完了した試行の遅延と未完了の増加の両方を含む．

隣接条件の人数差が等しくないとき，平均の差だけでは増加の速さを比べられない．人数差で割った傾きも有限回の推定値なので，32章の反復誤差の評価と組み合わせる．特定の行動が現実の停車時間を改善すると結論するには，行動の操作的定義と独立した検証が必要である．